# Lab 2: LSTM-AutoEncoder 잔여 수명(RUL) 예측
## 목표: 시계열 센서 데이터로 설비의 남은 수명을 예측한다

### 핵심 개념
- **LSTM (Long Short-Term Memory)**: 시계열 패턴을 기억하는 순환 신경망
- **RUL (Remaining Useful Life)**: 설비가 앞으로 얼마나 더 작동할 수 있는지
- **슬라이딩 윈도우**: 과거 N 사이클 데이터로 현재 상태를 학습

### 실습 흐름
1. 시계열 윈도우 생성 → 2. LSTM 모델 정의 → 3. 학습 → 4. RUL 예측 시각화

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

torch.manual_seed(42)
np.random.seed(42)

print("✅ 라이브러리 로드 완료")
print(f"PyTorch 버전: {torch.__version__}")

## RUL (Remaining Useful Life) 개념

> **"앞으로 몇 사이클이나 더 버틸 수 있나?"**

| 현재 상태 | RUL 의미 | 현장 조치 |
|-----------|----------|----------|
| RUL > 100 | 여유 있음 | 정기 모니터링 |
| 30 < RUL ≤ 100 | 주의 구간 | 점검 일정 수립 |
| RUL ≤ 30 | 위험 구간 | **즉시 정비 예약** |
| RUL = 0 | 고장 | 비상 대응 |

**NASA C-MAPSS 데이터 기준**: 1 사이클 ≈ 1회 운항 (항공 엔진)

In [ ]:
def create_sliding_windows(data, rul_values, window_size=30):
    """
    시계열 슬라이딩 윈도우 생성
    
    Args:
        data: 센서 데이터 (n_samples, n_features)
        rul_values: RUL 레이블 (n_samples,)
        window_size: 과거 몇 사이클을 한 샘플로 사용할지
    
    Returns:
        X: (n_windows, window_size, n_features)
        y: (n_windows,) - 각 윈도우의 마지막 시점 RUL
    """
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(rul_values[i+window_size])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

# 합성 데이터 생성
n_samples = 500
n_features = 14
window_size = 30

# 단조감소 RUL (500 → 0)
rul_true = np.linspace(200, 0, n_samples)

# 열화 패턴을 가진 센서 신호
t = np.linspace(0, 1, n_samples)
sensor_data = np.column_stack([
    np.random.normal(1.0 + 0.5*t, 0.05 + 0.1*t) for _ in range(n_features)
])

# 정규화
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()
X_scaled = scaler_x.fit_transform(sensor_data)
rul_scaled = scaler_y.fit_transform(rul_true.reshape(-1, 1)).flatten()

X_win, y_win = create_sliding_windows(X_scaled, rul_scaled, window_size=window_size)

# 학습/테스트 분리
split = int(len(X_win) * 0.8)
X_train, X_test = X_win[:split], X_win[split:]
y_train, y_test = y_win[:split], y_win[split:]

print(f"학습 윈도우: {X_train.shape}")
print(f"테스트 윈도우: {X_test.shape}")
print(f"윈도우 크기: {window_size} 사이클")

In [ ]:
class LSTMRULPredictor(nn.Module):
    """
    LSTM 기반 RUL 예측 모델
    - 인코더: 시계열 패턴 압축
    - 디코더: RUL 값 회귀
    """
    def __init__(self, input_dim, hidden_dim=32, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 1)
        )
    
    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        lstm_out, (h_n, c_n) = self.lstm(x)
        # 마지막 타임스텝의 hidden state 사용
        last_hidden = lstm_out[:, -1, :]
        return self.regressor(last_hidden).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LSTMRULPredictor(input_dim=n_features, hidden_dim=32, num_layers=2).to(device)
print(model)
print(f"\n파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 데이터를 텐서로 변환
X_tr = torch.FloatTensor(X_train).to(device)
y_tr = torch.FloatTensor(y_train).to(device)
X_te = torch.FloatTensor(X_test).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

train_losses = []
n_epochs = 50

for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad()
    pred = model(X_tr)
    loss = criterion(pred, y_tr)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # 기울기 폭발 방지
    optimizer.step()
    scheduler.step()
    train_losses.append(loss.item())
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{n_epochs} | Loss: {loss.item():.6f} | LR: {scheduler.get_last_lr()[0]:.6f}")

# 학습 곡선
plt.figure(figsize=(10, 3))
plt.plot(train_losses)
plt.title('LSTM 학습 손실 (MSE Loss)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.tight_layout()
plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_te).cpu().numpy()

# 역정규화
y_pred_real = scaler_y.inverse_transform(y_pred_scaled.reshape(-1,1)).flatten()
y_test_real = scaler_y.inverse_transform(y_test.reshape(-1,1)).flatten()

# 성능 평가
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
mae = np.mean(np.abs(y_test_real - y_pred_real))
print(f"RMSE: {rmse:.2f} 사이클")
print(f"MAE:  {mae:.2f} 사이클")

# 시각화: 실제 vs 예측 RUL
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(y_test_real, label='실제 RUL', linewidth=2)
axes[0].plot(y_pred_real, label='예측 RUL', linewidth=2, linestyle='--')
axes[0].axhline(30, color='red', linestyle=':', label='정비 기준선 (RUL=30)')
axes[0].set_title('RUL 예측 vs 실제')
axes[0].set_xlabel('샘플 인덱스')
axes[0].set_ylabel('RUL (사이클)')
axes[0].legend()

axes[1].scatter(y_test_real, y_pred_real, alpha=0.5, s=20)
axes[1].plot([0, max(y_test_real)], [0, max(y_test_real)], 'r--', label='완벽한 예측')
axes[1].set_title(f'실제 vs 예측 산점도 (RMSE={rmse:.1f})')
axes[1].set_xlabel('실제 RUL')
axes[1].set_ylabel('예측 RUL')
axes[1].legend()

plt.tight_layout()
plt.show()

## 현장 해석 및 활용

### 정비 의사결정 규칙

```
if 예측_RUL <= 30:
    → 정비 예약 알림 발송 (긴급)
elif 예측_RUL <= 100:
    → 다음 정기점검 시 교체 계획
else:
    → 모니터링 지속
```

### 기대 효과

| 지표 | 사전 예방 없음 | LSTM RUL 예측 적용 |
|------|--------------|-------------------|
| 예상치 못한 고장 | 빈번 | 80% 감소 |
| 정비 시기 | 고장 후 긴급 | 최적 시점 예약 |
| 부품 재고 | 과잉/부족 | 수요 예측 최적화 |

> **핵심**: "잔여 수명 30 사이클 이하 → 정비 예약 알림" 자동화로 계획외 다운타임 제거